# 02 — Corpus Build

Thin notebook (PLAN.md §6): runs the T1.8 end-to-end pipeline
(`cragb.data.build_corpus`) and displays the resulting `corpus_v1`
composition — it contains no pipeline logic of its own, only calls into
`src/cragb` and renders the manifest/reports that call already produced.

Re-running this notebook re-runs the *entire* build (raw files -> stream
-> clean -> join -> stratify -> write parquet), which takes on the order
of 15-20 minutes (dominated by language detection over the candidate
pool and one streamed pass over the metadata file). It is safe to re-run:
the seed in `configs/data.yaml` makes the output deterministic.

In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd

from cragb.data.build_corpus import build_corpus
from cragb.utils.io import load_config, resolve_path

cfg = load_config("configs/data.yaml")
cfg["paths"], cfg["sampling"]

({'raw_dir': 'data/raw',
  'processed_dir': 'data/processed',
  'reviews_raw': 'data/raw/reviews_clothing.jsonl.gz',
  'metadata_raw': 'data/raw/meta_clothing.jsonl.gz',
  'raw_checksums': 'data/raw/checksums.json',
  'corpus_out': 'data/processed/corpus_v1.parquet',
  'manifest_out': 'data/processed/corpus_v1_manifest.json'},
 {'target_size': 200000,
  'chunk_size': 20000,
  'overcollect_factor': 1.5,
  'strata': {'rating': {'1-2': 0.2, '3': 0.15, '4-5': 0.65},
   'has_image': {True: 0.15, False: 0.85}}})

## Run the pipeline

In [2]:
manifest = build_corpus("configs/data.yaml")
manifest["row_count"], manifest["sha256"][:16]

(200000, '61d02e5d7d7df570')

## Manifest — full pipeline report

In [3]:
import json
print(json.dumps(manifest, indent=2))

{
  "created_at_utc": "2026-07-21T18:03:24.969395+00:00",
  "seed": 42,
  "row_count": 200000,
  "sha256": "61d02e5d7d7df5701b84f9a077d273dc7d8a10a572005380d8ee88be43165557",
  "columns": {
    "rating": "float64",
    "title": "object",
    "text": "object",
    "asin": "object",
    "parent_asin": "object",
    "user_id": "object",
    "timestamp": "int64",
    "helpful_vote": "int64",
    "verified_purchase": "bool",
    "rating_bucket": "object",
    "has_image": "bool",
    "n_images": "int64",
    "image_urls": "object",
    "token_len": "int64",
    "product_title": "object",
    "main_category": "object",
    "categories": "object",
    "store": "object",
    "average_rating": "float64",
    "price": "float64"
  },
  "config_path": "configs/data.yaml",
  "reports": {
    "collect_candidate_pool": {
      "rows_scanned": 1560000,
      "scan_seconds": 19.2,
      "targets": {
        "1-2|True": 6000,
        "1-2|False": 34000,
        "3|True": 4500,
        "3|False": 25500,


## Load the frozen corpus and sanity-check composition

In [ ]:
corpus = pd.read_parquet(resolve_path(cfg["paths"]["corpus_out"]))
print(corpus.shape)
corpus.dtypes

In [ ]:
print("rating_bucket distribution (target 1-2=0.20, 3=0.15, 4-5=0.65):")
print(corpus["rating_bucket"].value_counts(normalize=True))
print()
print("has_image distribution (target True=0.15, False=0.85):")
print(corpus["has_image"].value_counts(normalize=True))
print()
print("null rate per column:")
print((corpus.isna().mean() * 100).round(2))

## Integrity check

Recompute the parquet file's SHA-256 independently and confirm it matches
the manifest — proves the manifest describes the file actually on disk,
not a stale record from a previous run.

In [ ]:
from cragb.utils.io import sha256_file

recomputed = sha256_file(cfg["paths"]["corpus_out"])
assert recomputed == manifest["sha256"], "corpus_v1.parquet does not match its manifest hash!"
print("OK: corpus_v1.parquet matches manifest sha256:", recomputed[:16])